## ⚙️ **Libraries Import**

In [108]:
# Set seed for reproducibility
SEED = 45

# Import necessary libraries
import os

# Set environment variables before importing modules
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['MPLCONFIGDIR'] = os.getcwd() + '/configs/'

# Suppress warnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=Warning)

# Import necessary modules
import logging
import random
import numpy as np

# Set seeds for random number generators in NumPy and Python
np.random.seed(SEED)
random.seed(SEED)

# Import PyTorch
import torch
torch.manual_seed(SEED)
from torch import nn
# from torchsummary import summary
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import TensorDataset, DataLoader
logs_dir = "tensorboard"
!pkill -f tensorboard
%load_ext tensorboard
!mkdir -p models

if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
else:
    device = torch.device("cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

# Import other libraries
import copy
import shutil
from itertools import product
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configure plot display settings
sns.set(font_scale=1.4)
sns.set_style('white')
plt.rc('font', size=14)
%matplotlib inline

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard
PyTorch version: 2.6.0+cu124
Device: cuda


## ⏳ **Data Loading**

In [109]:
os.environ["DATASETS"] = "."
os.environ["DATASET_train"] = "/kaggle/input/the-pirate-pain-dataset/pirate_pain_train.csv"
os.environ["DATASET_test"] = "/kaggle/input/the-pirate-pain-dataset/pirate_pain_test.csv"
os.environ["DATASET_train-labels"] = "/kaggle/input/the-pirate-pain-dataset/pirate_pain_train_labels.csv"

# Load the dataset from a CSV file
df_train = pd.read_csv(os.environ["DATASET_train"])
df_public_test = pd.read_csv(os.environ["DATASET_test"])
df_labels = pd.read_csv(os.environ["DATASET_train-labels"])

## 🔄 **Data Preprocessing**

In [110]:
from sklearn.preprocessing import MinMaxScaler
def preProcess(df):
    df["pain_survey"] = df[["pain_survey_1" ,"pain_survey_2", "pain_survey_3", "pain_survey_4"]].min(axis=1)
    df.drop(columns=['pain_survey_1', 'pain_survey_2', 'pain_survey_3', 'pain_survey_4'], inplace=True)
    df['merged_n_features'] = np.where(
        (df['n_legs'] == 'two') & (df['n_hands'] == 'two') & (df['n_eyes'] == 'two'),
        0, 
        1  
    )
    
    # Calculate the count of rows where n_legs, n_hands, or n_eyes are not 'two'
    explicit_different_count = df[
        (df['n_legs'] != 'two') | (df['n_hands'] != 'two') | (df['n_eyes'] != 'two')
    ].shape[0]
    
    df.drop(columns=['n_legs', 'n_hands', 'n_eyes', "joint_30"], inplace=True)
    
    columns_to_scale = [col for col in df.columns if col not in ['time', 'sample_index']]

    scaler = MinMaxScaler()
    df[columns_to_scale] = scaler.fit_transform(df[columns_to_scale])

preProcess(df_train)
preProcess(df_public_test)

In [111]:
from sklearn.model_selection import train_test_split

# Split users into train and val, stratified by label (means that each class will be
# represented in the same proportion in both train and val)
train_users, val_users = train_test_split(
    df_labels['sample_index'],
    test_size=0.2,              # 20% validation
    stratify=df_labels['label'],
    random_state=SEED
)

# Select rows for train and validation
df_train_fold = df_train[df_train['sample_index'].isin(train_users)]
df_val_fold   = df_train[df_train['sample_index'].isin(val_users)]

# Merge labels into the training dataframe
df_train_fold = df_train_fold.merge(df_labels, on='sample_index', how='left')
df_val_fold = df_val_fold.merge(df_labels, on='sample_index', how='left')

# Check class distribution
print(df_train_fold['label'].value_counts(normalize=True))
print(df_val_fold['label'].value_counts(normalize=True))

df_train = df_train_fold.copy()
df_train = df_train.drop('label', axis=1)
df_val = df_val_fold.copy()
df_val = df_val.drop('label', axis=1)

label
no_pain      0.772727
low_pain     0.142045
high_pain    0.085227
Name: proportion, dtype: float64
label
no_pain      0.774436
low_pain     0.142857
high_pain    0.082707
Name: proportion, dtype: float64


In [112]:
# Map int64, float64, string to float32
mapping = {
    "one+peg_leg": 1,
    "one+hook_hand": 1,
    "one+eye_patch": 1,
    "two": 2,
}

# Train data
for col in df_train:
    if df_train[col].dtype == 'int64' and col not in ['sample_index', 'time']:
        df_train[col] = df_train[col].astype(np.float32)
    elif df_train[col].dtype == 'float64':
        df_train[col] = df_train[col].astype(np.float32)
    elif df_train[col].dtype == 'object':
        df_train[col] = df_train[col].replace(mapping)
        df_train[col] = df_train[col].astype(np.float32)

#Validation data
for col in df_val:
    if df_val[col].dtype == 'int64' and col not in ['sample_index', 'time']:
        df_val[col] = df_val[col].astype(np.float32)
    elif df_val[col].dtype == 'float64':
        df_val[col] = df_val[col].astype(np.float32)
    elif df_val[col].dtype == 'object':
        df_val[col] = df_val[col].replace(mapping)
        df_val[col] = df_val[col].astype(np.float32)
        
#Public test data
for col in df_public_test:
    if df_public_test[col].dtype == 'int64' and col not in ['sample_index', 'time']:
        df_public_test[col] = df_public_test[col].astype(np.float32)
    elif df_public_test[col].dtype == 'float64':
        df_public_test[col] = df_public_test[col].astype(np.float32)
    elif df_public_test[col].dtype == 'object':
        df_public_test[col] = df_public_test[col].replace(mapping)
        df_public_test[col] = df_public_test[col].astype(np.float32)

In [113]:
# Map labels
label_mapping = {label: idx for idx, label in enumerate(df_labels['label'].unique())}
df_labels['label_idx'] = df_labels['label'].map(label_mapping)
df_labels.drop('label', axis=1, inplace=True)

In [114]:
# List of feature columns
exclude = ["sample_index", "time", 'n_legs', 'n_hands', 'n_eyes']
feature_cols = [col for col in df_train.columns if col not in exclude]

## 🛠️ **Model Building**

In [115]:
class RecurrentClassifierWithStaticBranch(nn.Module):
    """
    LSTM classifier with separate branch for static features.
    This is the recommended approach as it's more efficient and allows
    better representation learning for static vs temporal features.
    """
    def __init__(
            self,
            temporal_input_size,    # Size of temporal features
            static_input_size,      # Size of static features
            hidden_size,
            num_layers,
            num_classes,
            static_hidden_sizes=[32, 16],  # MLP layers for static features
            rnn_type='GRU',
            bidirectional=False,
            dropout_rate=0.2
            ):
        super().__init__()

        self.rnn_type = rnn_type
        self.num_layers = num_layers
        self.hidden_size = hidden_size
        self.bidirectional = bidirectional

        # Map string name to PyTorch RNN class
        rnn_map = {
            'RNN': nn.RNN,
            'LSTM': nn.LSTM,
            'GRU': nn.GRU
        }

        if rnn_type not in rnn_map:
            raise ValueError("rnn_type must be 'RNN', 'LSTM', or 'GRU'")

        rnn_module = rnn_map[rnn_type]

        # Dropout for RNN (only between layers if num_layers > 1)
        dropout_val = dropout_rate if num_layers > 1 else 0

        # Recurrent layer for temporal features
        self.rnn = rnn_module(
            input_size=temporal_input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout_val
        )

        # MLP branch for static features
        static_layers = []
        input_size = static_input_size
        for hidden_size_static in static_hidden_sizes:
            static_layers.append(nn.Linear(input_size, hidden_size_static))
            static_layers.append(nn.ReLU())
            static_layers.append(nn.Dropout(dropout_rate))
            input_size = hidden_size_static
        self.static_branch = nn.Sequential(*static_layers)
        static_output_size = static_hidden_sizes[-1]

        # Calculate input size for the final classifier
        if self.bidirectional:
            rnn_output_size = hidden_size * 2
        else:
            rnn_output_size = hidden_size

        # Final classification layer (combines RNN output + static branch output)
        classifier_input_size = rnn_output_size + static_output_size
        self.classifier = nn.Linear(classifier_input_size, num_classes)

    def forward(self, x_temporal, x_static):
        """
        x_temporal: (batch_size, seq_length, temporal_input_size)
        x_static: (batch_size, static_input_size)
        """
        # Process temporal features through RNN
        rnn_out, hidden = self.rnn(x_temporal)

        # Extract last hidden state
        if self.rnn_type == 'LSTM':
            hidden = hidden[0]

        if self.bidirectional:
            hidden = hidden.view(self.num_layers, 2, -1, self.hidden_size)
            rnn_output = torch.cat([hidden[-1, 0, :, :], hidden[-1, 1, :, :]], dim=1)
        else:
            rnn_output = hidden[-1]  # (batch_size, hidden_size)

        # Process static features through MLP branch
        static_output = self.static_branch(x_static)  # (batch_size, static_hidden_sizes[-1])

        # Concatenate RNN output and static branch output
        combined = torch.cat([rnn_output, static_output], dim=1)  # (batch_size, rnn_output_size + static_output_size)

        # Get logits
        logits = self.classifier(combined)
        return logits


In [116]:
def build_sequences_with_static(df, df_labels, window, stride, temporal_cols, static_cols):
    """
    Build sliding window sequences WITH static features separated.
    
    Parameters
    ----------
    df : DataFrame
        Feature data (contains sample_index, time, and feature columns)
    df_labels : DataFrame
        Label data (sample_index, label_idx)
    window : int
        Number of timesteps per sequence
    stride : int
        Step between windows (controls overlap)
    temporal_cols : list
        List of temporal feature column names (e.g., joint features)
    static_cols : list
        List of static feature column names (e.g., n_legs, n_hands, n_eyes)
    
    Returns
    -------
    X_temporal : np.array   shape = (num_sequences, window, num_temporal_features)
    X_static : np.array     shape = (num_sequences, num_static_features)
    y : np.array            shape = (num_sequences,)
    """
    
    
    X_temporal = []
    X_static = []
    y = []
    
    for sid in df["sample_index"].unique():
        # Extract temporal features per sequence
        temp_temporal = df[df["sample_index"] == sid][temporal_cols].values.astype("float32")
        
        # Check and handle NaN values in temporal features
        if np.isnan(temp_temporal).any():
            # Forward fill, then backward fill, then fill remaining with 0
            temp_df = pd.DataFrame(temp_temporal, columns=temporal_cols)
            temp_df = temp_df.ffill().bfill().fillna(0)
            temp_temporal = temp_df.values.astype("float32")
        
        # Extract static features (same for all timesteps, take first row)
        temp_static = df[df["sample_index"] == sid][static_cols].values[0].astype("float32")
        
        # Check and handle NaN values in static features
        if np.isnan(temp_static).any():
            temp_static = np.nan_to_num(temp_static, nan=0.0)
        
        # Get label
        label = df_labels.loc[df_labels["sample_index"] == sid, "label_idx"].iloc[0]
        
        # Padding section
        remainder = len(temp_temporal) % window
        if remainder > 0:
            pad_len = window - remainder
            padding = np.zeros((pad_len, temp_temporal.shape[1]), dtype="float32")
            temp_temporal = np.concatenate((temp_temporal, padding), axis=0)
        
        # Build sliding windows
        idx = 0
        while idx + window <= len(temp_temporal):
            X_temporal.append(temp_temporal[idx:idx + window])
            X_static.append(temp_static)  # Static features are the same for all windows from same sample
            y.append(label)
            idx += stride
    
    return np.array(X_temporal), np.array(X_static), np.array(y)


# Example usage:
# Define which features are temporal vs static
temporal_feature_cols = [col for col in df_train.columns 
                         if col not in ["sample_index", "time", "pain_survey", "merged_n_features"] 
                         and not col.startswith('n_')]  # Joint features only

static_feature_cols = ['pain_survey', 'merged_n_features']  # Static features

print(f"Temporal features: {len(temporal_feature_cols)}")
print(f"Static features: {len(static_feature_cols)}")
print(f"Temporal: {temporal_feature_cols[:5]}...")
print(f"Static: {static_feature_cols}")


Temporal features: 30
Static features: 2
Temporal: ['joint_00', 'joint_01', 'joint_02', 'joint_03', 'joint_04']...
Static: ['pain_survey', 'merged_n_features']


In [117]:
class TemporalStaticDataset(torch.utils.data.Dataset):
    """
    Custom Dataset that handles both temporal and static features.
    """
    def __init__(self, X_temporal, X_static, y):
        """
        Args:
            X_temporal: np.array of shape (num_samples, seq_len, num_temporal_features)
            X_static: np.array of shape (num_samples, num_static_features)
            y: np.array of shape (num_samples,)
        """
        self.X_temporal = torch.from_numpy(X_temporal).float()
        self.X_static = torch.from_numpy(X_static).float()
        self.y = torch.from_numpy(y).long()
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X_temporal[idx], self.X_static[idx], self.y[idx]

In [118]:
def train_one_epoch_with_static(model, train_loader, criterion, optimizer, scaler, device, l1_lambda=0, l2_lambda=0):
    """
    Modified training function to handle models with static features.
    """
    model.train()
    
    running_loss = 0.0
    all_predictions = []
    all_targets = []
    
    for batch_idx, (inputs_temporal, inputs_static, targets) in enumerate(train_loader):
        inputs_temporal = inputs_temporal.to(device)
        inputs_static = inputs_static.to(device)
        targets = targets.to(device)
        
        # Check for NaN in inputs
        if torch.isnan(inputs_temporal).any() or torch.isnan(inputs_static).any():
            print(f"Warning: NaN in inputs at batch {batch_idx}, skipping")
            continue
        
        optimizer.zero_grad(set_to_none=True)
        
        with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
            # Forward pass with both temporal and static features
            logits = model(inputs_temporal, inputs_static)
            
            # Check for NaN in logits
            if torch.isnan(logits).any():
                print(f"Warning: NaN in logits at batch {batch_idx}")
                continue
            
            loss = criterion(logits, targets)
            
            # Check for NaN in loss
            if torch.isnan(loss) or torch.isinf(loss):
                print(f"Warning: Invalid loss at batch {batch_idx}: {loss.item()}")
                continue
            
            # Check again after regularization
            if torch.isnan(loss) or torch.isinf(loss):
                print(f"Warning: Invalid loss after regularization at batch {batch_idx}")
                continue
        
        scaler.scale(loss).backward()
        
        # Gradient clipping to prevent explosion
        if device.type == 'cuda':
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            # For CPU/MPS, clip gradients before optimizer step
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        
        running_loss += loss.item() * inputs_temporal.size(0)
        predictions = logits.argmax(dim=1)
        all_predictions.append(predictions.cpu().numpy())
        all_targets.append(targets.cpu().numpy())
    
    epoch_loss = running_loss / len(train_loader.dataset)
    
    # Calculate F1 score with error handling
    all_targets_arr = np.concatenate(all_targets)
    all_predictions_arr = np.concatenate(all_predictions)
    
    # Check for NaN or invalid values
    if np.isnan(epoch_loss) or np.isinf(epoch_loss):
        print(f"Warning: Invalid loss value: {epoch_loss}")
        epoch_f1 = 0.0
    else:
        try:
            epoch_f1 = f1_score(all_targets_arr, all_predictions_arr, average='macro', zero_division=0)
            if np.isnan(epoch_f1):
                epoch_f1 = 0.0
        except:
            epoch_f1 = 0.0
    
    return epoch_loss, epoch_f1


def validate_one_epoch_with_static(model, val_loader, criterion, device):
    """
    Modified validation function to handle models with static features.
    """
    model.eval()
    
    running_loss = 0.0
    all_predictions = []
    all_targets = []
    
    with torch.no_grad():
        for inputs_temporal, inputs_static, targets in val_loader:
            inputs_temporal = inputs_temporal.to(device)
            inputs_static = inputs_static.to(device)
            targets = targets.to(device)
            
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                logits = model(inputs_temporal, inputs_static)
                loss = criterion(logits, targets)
            
            running_loss += loss.item() * inputs_temporal.size(0)
            predictions = logits.argmax(dim=1)
            all_predictions.append(predictions.cpu().numpy())
            all_targets.append(targets.cpu().numpy())
    
    epoch_loss = running_loss / len(val_loader.dataset)
    
    # Calculate F1 score with error handling
    all_targets_arr = np.concatenate(all_targets)
    all_predictions_arr = np.concatenate(all_predictions)
    
    # Check for NaN or invalid values
    if np.isnan(epoch_loss) or np.isinf(epoch_loss):
        print(f"Warning: Invalid loss value: {epoch_loss}")
        epoch_accuracy = 0.0
    else:
        try:
            epoch_accuracy = f1_score(all_targets_arr, all_predictions_arr, average='macro', zero_division=0)
            if np.isnan(epoch_accuracy):
                epoch_accuracy = 0.0
        except:
            epoch_accuracy = 0.0
    
    return epoch_loss, epoch_accuracy


In [119]:
def fit_with_static(model, train_loader, val_loader, epochs, criterion, optimizer, scaler, device, scheduler,
                    l1_lambda=0, l2_lambda=0, patience=10, verbose=1):
    history = {'train_loss': [], 'val_loss': [], 'train_f1': [], 'val_f1': []}
    best_loss, patience_cnt = 1000, 0
    
    for epoch in range(1, epochs + 1):
        train_loss, train_f1 = train_one_epoch_with_static(
            model, train_loader, criterion, optimizer, scaler, device, l1_lambda, l2_lambda
        )
        val_loss, val_f1 = validate_one_epoch_with_static(model, val_loader, criterion, device)
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_f1'].append(train_f1)
        history['val_f1'].append(val_f1)
        

        if val_loss <= best_loss:
            best_loss = val_loss
            torch.save(model.state_dict(), "models/lstm_static_best.pt")
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= patience:
                print(f"Early stopping at epoch {epoch}")
                break
        
        if scheduler is not None:
            # We use the metric defined by 'evaluation_metric' to step the scheduler.
            metric_to_monitor = training_history[evaluation_metric][-1]
            scheduler.step(metric_to_monitor)
        
        if verbose > 0 and epoch % verbose == 0:
            print(f"Epoch {epoch}: Train Loss={train_loss}, Val Loss={val_loss}")
            print(f"Epoch {epoch}: Train F1={train_f1:.4f}, Val F1={val_f1:.4f}")
    
    model.load_state_dict(torch.load("models/lstm_static_best.pt"))
    return model, history

## Implementation of the Grid Search

In [120]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    """
    Focal Loss for multi-class classification.
    Reference: https://arxiv.org/abs/1708.02002 (Lin et al. 2017)

    Args:
        alpha (float or list): Weighting factor for classes (balances class imbalance)
        gamma (float): Focusing parameter to reduce the loss for well-classified examples
        reduction (str): 'none' | 'mean' | 'sum'
    """
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        if isinstance(alpha, (float, int)):
            self.alpha = torch.tensor([alpha])
        else:
            self.alpha = torch.tensor(alpha) if alpha is not None else None
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        """
        Args:
            inputs: Predictions (logits), shape [batch_size, num_classes]
            targets: Ground truth labels, shape [batch_size]
        """
        # Compute log-probabilities
        log_probs = F.log_softmax(inputs, dim=1)
        probs = torch.exp(log_probs)

        # Select log-probability of the correct class
        log_probs_true = log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        probs_true = probs.gather(1, targets.unsqueeze(1)).squeeze(1)

        # Compute focal weight
        focal_weight = (1 - probs_true) ** self.gamma

        # Apply alpha (class weight)
        if self.alpha is not None:
            if self.alpha.device != inputs.device:
                self.alpha = self.alpha.to(inputs.device)
            alpha_factor = self.alpha[targets]
            focal_weight = alpha_factor * focal_weight

        # Compute final loss
        loss = -focal_weight * log_probs_true

        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        else:
            return loss

In [121]:
!pip install torch_optimizer --quiet

^C


In [122]:
import math
import torch
from torch.optim import Optimizer

class Ranger(Optimizer):
    def __init__(self, params, lr=1e-3, alpha=0.5, k=6, betas=(0.95, 0.999), eps=1e-5, weight_decay=0):
        """
        Ranger = RAdam + Lookahead
        Args:
            params: model parameters
            lr: learning rate
            alpha: lookahead step size (0.5 is default)
            k: lookahead steps before sync (6 is default)
            betas: RAdam betas
            eps: numerical stability
            weight_decay: L2 regularization
        """
        defaults = dict(lr=lr, alpha=alpha, k=k, betas=betas, eps=eps, weight_decay=weight_decay)
        super(Ranger, self).__init__(params, defaults)
        self._step = 0

        for group in self.param_groups:
            group["slow_params"] = [p.clone().detach() for p in group["params"] if p.requires_grad]

    def step(self, closure=None):
        loss = None
        if closure is not None:
            loss = closure()

        for group in self.param_groups:
            for p, sp in zip(group["params"], group["slow_params"]):
                if p.grad is None:
                    continue

                grad = p.grad.data
                if grad.is_sparse:
                    raise RuntimeError("Ranger does not support sparse gradients")

                state = self.state[p]

                # State initialization
                if len(state) == 0:
                    state["step"] = 0
                    state["exp_avg"] = torch.zeros_like(p.data)
                    state["exp_avg_sq"] = torch.zeros_like(p.data)

                exp_avg, exp_avg_sq = state["exp_avg"], state["exp_avg_sq"]
                beta1, beta2 = group["betas"]

                state["step"] += 1
                self._step += 1

                # Apply weight decay
                if group["weight_decay"] != 0:
                    grad = grad.add(p.data, alpha=group["weight_decay"])

                # Update exponential moving averages
                exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)
                exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)

                # Compute rectified term (RAdam)
                bias_correction1 = 1 - beta1 ** state["step"]
                bias_correction2 = 1 - beta2 ** state["step"]
                n_sma_max = 2 / (1 - beta2) - 1
                n_sma = n_sma_max - 2 * state["step"] * (beta2 ** state["step"]) / bias_correction2

                if n_sma >= 5:
                    step_size = group["lr"] * math.sqrt(
                        ((1 - beta2 ** state["step"]) * (n_sma - 4) / (n_sma_max - 4)) *
                        ((n_sma - 2) / n_sma) * (n_sma_max / (n_sma_max - 2))
                    ) / bias_correction1
                    denom = exp_avg_sq.sqrt().add_(group["eps"])
                    p.data.addcdiv_(exp_avg, denom, value=-step_size)
                else:
                    step_size = group["lr"] / bias_correction1
                    p.data.add_(exp_avg, alpha=-step_size)

                # Lookahead updates
                if self._step % group["k"] == 0:
                    sp.add_(p.data - sp, alpha=group["alpha"])
                    p.data.copy_(sp)

        return loss


In [123]:
import torch
import torch.nn as nn
import numpy as np
from itertools import product
from torch.utils.data import DataLoader
from typing import Dict, Any
from torch.optim.lr_scheduler import ReduceLROnPlateau
def grid_search_cv_rnn(df: Any, df_val: Any, device: torch.device, temporal_feature_cols: list, static_feature_cols: list,
                       window_size: int, stride: int, df_label: str,
                       param_grid: Dict[str, list], fixed_params: Dict[str, Any], cv_params: Dict[str, Any],
                       evaluation_metric: str = "val_f1", mode: str = 'max',
                       restore_best_weights: bool = True, verbose: int = 10, seed: int = 42,
                       experiment_name: str = "") -> tuple:
    """
    Execute grid search by performing a SINGLE train/validation run for each configuration.

    Args:
        df: DataFrame containing the time series data.
        device: PyTorch device (e.g., 'cuda' or 'cpu').
        temporal_feature_cols: List of column names for temporal features.
        static_feature_cols: List of column names for static features.
        window_size: Size of the time window for sequence building.
        stride: Stride used for generating sequences.
        df_label: The column name for the target label.
        param_grid: Dict of hyperparameters (keys) and lists of values (values) to test.
        fixed_params: Dict of fixed hyperparameters used across all configurations.
        cv_params: Dict of CV settings controlling the training run (EPOCHS, PATIENCE, VERBOSE).
        evaluation_metric: The metric to use for tracking the best configuration (e.g., 'val_f1').
        mode: 'max' (higher is better) or 'min' (lower is better) for the evaluation metric.
        restore_best_weights: Whether to restore the best model weights during the single run.
        verbose: Print progress for each configuration.
        seed: Random seed for reproducibility.
        experiment_name: Name for the experiment (for logging purposes).

    Returns:
        results: Dict with scores for each configuration.
        best_config: Dict with best hyperparameter combination.
        best_score: Best final validation score achieved.
    """
    # ----------------------------------------------
    # 1. Setup Grid Search Combinations
    # ----------------------------------------------
    param_names = list(param_grid.keys())
    param_values = list(param_grid.values())
    combinations = list(product(*param_values)) # all the possible combinations
    best_model = None
    results = {}
    best_score = -np.inf if mode == 'max' else np.inf
    best_config = None

    total = len(combinations)

    print("Preparing sequence data (Train/Validation Split)...")

  # Corrected: Pass 'train' and 'val' split indicators instead of 'df_label'
    X_train_temporal, X_train_static, y_train_static = build_sequences_with_static(
        df, df_labels, window_size, stride, temporal_feature_cols, static_feature_cols
    )
    X_val_temporal, X_val_static, y_val_static = build_sequences_with_static(
        df_val, df_labels, window_size, stride, temporal_feature_cols, static_feature_cols
    )


    # Calculate class weights once
    num_classes_static = len(np.unique(y_train_static))
    class_counts = np.bincount(y_train_static, minlength=num_classes_static)
    class_weights = 1.0 / (class_counts + 1e-8)
    weights = torch.tensor(class_weights / class_weights.sum(), dtype=torch.float32).to(device)
    
    # ----------------------------------------------
    # 3. Start Grid Search Loop
    # ----------------------------------------------
    for idx, combo in enumerate(combinations, 1):
        # Create current configuration dict
        current_config = dict(zip(param_names, combo))
        config_str = "_".join([f"{k}_{v}" for k, v in current_config.items()])

        if verbose:
            print(f"\n--- Running Configuration {idx}/{total}: {config_str} ---")

        # Merge current config with fixed parameters to create a single run parameter set
        run_params = {**fixed_params, **current_config}

        # 3.1. Prepare DataLoader for Current Config
        train_ds_static = TemporalStaticDataset(X_train_temporal, X_train_static, y_train_static)
        val_ds_static = TemporalStaticDataset(X_val_temporal, X_val_static, y_val_static)
        
        BATCH_SIZE = run_params['batch_size']
        train_loader_static = DataLoader(train_ds_static, batch_size=BATCH_SIZE, shuffle=True, drop_last=False, num_workers=0)
        val_loader_static = DataLoader(val_ds_static, batch_size=BATCH_SIZE, shuffle=False, drop_last=False, num_workers=0)

        # 3.2. Initialize Model, Criterion, Optimizer
        
        model_with_static = RecurrentClassifierWithStaticBranch(
            temporal_input_size=X_train_temporal.shape[2],
            static_input_size=X_train_static.shape[1],
            hidden_size=run_params['hidden_size'],
            num_layers=run_params['hidden_layers'],
            num_classes=num_classes_static,
            static_hidden_sizes=run_params.get('static_hidden_sizes', [32, 16]), # Assumed structure
            rnn_type=run_params['rnn_type'],
            bidirectional=run_params['bidirectional'],
            dropout_rate=run_params['dropout_rate']
        ).to(device)

        # Criterion (Using pre-calculated weights)
        criterion = FocalLoss(alpha=class_weights, gamma=0.5, reduction='mean')
        scaler_static = torch.amp.GradScaler(enabled=(device.type == 'cuda'))

        # Optimizer
        optimizer_static = torch.optim.AdamW(
            model_with_static.parameters(), 
            lr=run_params['learning_rate'], 
            weight_decay=run_params['l2_lambda']
        )
        optimizer = Ranger(
            model_with_static.parameters(),
            lr=run_params['learning_rate'],
            weight_decay=run_params['l2_lambda']
)
        
        # 3.3. Execute Single Training Run
        model, history = fit_with_static(
            model=model_with_static, 
            train_loader=train_loader_static, 
            val_loader=val_loader_static, 
            epochs=cv_params['EPOCHS'],
            criterion=criterion, 
            optimizer=optimizer_static, 
            scaler=scaler_static,
            device=device,
            scheduler = None,
            l1_lambda=run_params['l1_lambda'], 
            l2_lambda=run_params['l2_lambda'], 
            patience=cv_params['PATIENCE'], 
            verbose=cv_params['VERBOSE']
        )
        

        # 3.4. Extract and Track Score
        final_val_score = max(history[evaluation_metric])
        if verbose:
            print(f"--- Configuration Summary: Final Val {evaluation_metric}: {final_val_score:.4f} ---")

        # Store results
        results[config_str] = {evaluation_metric: final_val_score}

        # Track best configuration
        is_better = (mode == 'max' and final_val_score > best_score) or \
                    (mode == 'min' and final_val_score < best_score)

        if is_better:
            best_model = model
            best_score = final_val_score
            best_config = current_config.copy()
            if verbose:
                print("  => NEW BEST SCORE!")

    return results, best_config, best_score, best_model

In [124]:
WINDOW_SIZE = 100
STRIDE = 50

In [125]:
param_grid = {
    'hidden_size': [64,128],
    'learning_rate': [1e-3],
    'batch_size': [16, 32],
    'hidden_layers': [2,3]
}

fixed_params = {
    'rnn_type': 'GRU',
    'bidirectional': True,
    'l1_lambda': 0.0,
    'l2_lambda': 1e-3,
    'dropout_rate':  0.338 ,
    'static_hidden_sizes': [64, 32, 16] 
}

cv_params = {
    'EPOCHS': 200,
    'PATIENCE': 10, # Early stopping patience
    'VERBOSE': 1 # Verbosity for the training loop itself
}


results, best_config, best_score, best_model = grid_search_cv_rnn(
    df=df_train, 
    df_val = df_val,
    device=device, 
    temporal_feature_cols=temporal_feature_cols, 
    static_feature_cols=static_feature_cols,
    window_size=WINDOW_SIZE, 
    stride=STRIDE, 
    df_label=df_labels,
    
    param_grid=param_grid, 
    fixed_params=fixed_params, 
    cv_params=cv_params,
        
    evaluation_metric="val_f1", 
    mode='max', 
    verbose=1 # Verbosity for the grid search progress
)


Preparing sequence data (Train/Validation Split)...

--- Running Configuration 1/8: hidden_size_64_learning_rate_0.001_batch_size_16_hidden_layers_2 ---
Epoch 1: Train Loss=0.0016955929306219093, Val Loss=0.0016744854204266111
Epoch 1: Train F1=0.2907, Val F1=0.1131
Epoch 2: Train Loss=0.0015976217038335882, Val Loss=0.0016143488267423129
Epoch 2: Train F1=0.3887, Val F1=0.2459
Epoch 3: Train Loss=0.0015049083063499904, Val Loss=0.0015929545703154513
Epoch 3: Train F1=0.4300, Val F1=0.4295
Epoch 4: Train Loss=0.001503150563039787, Val Loss=0.0015963569297294603
Epoch 4: Train F1=0.4285, Val F1=0.4545
Epoch 5: Train Loss=0.0013661484718949484, Val Loss=0.0016050876113449348
Epoch 5: Train F1=0.4646, Val F1=0.4971
Epoch 6: Train Loss=0.0013082616045803008, Val Loss=0.0015465955809732233
Epoch 6: Train F1=0.4796, Val F1=0.4448
Epoch 7: Train Loss=0.0012861621003323251, Val Loss=0.0014845507266511755
Epoch 7: Train F1=0.5173, Val F1=0.5072
Epoch 8: Train Loss=0.0012138197944804975, Val Los

In [126]:
(results, best_config, best_score)

({'hidden_size_64_learning_rate_0.001_batch_size_16_hidden_layers_2': {'val_f1': 0.9035401377790354},
  'hidden_size_64_learning_rate_0.001_batch_size_16_hidden_layers_3': {'val_f1': 0.8151616867980506},
  'hidden_size_64_learning_rate_0.001_batch_size_32_hidden_layers_2': {'val_f1': 0.8628921172183116},
  'hidden_size_64_learning_rate_0.001_batch_size_32_hidden_layers_3': {'val_f1': 0.868678794923956},
  'hidden_size_128_learning_rate_0.001_batch_size_16_hidden_layers_2': {'val_f1': 0.8570649813644465},
  'hidden_size_128_learning_rate_0.001_batch_size_16_hidden_layers_3': {'val_f1': 0.8937297886398374},
  'hidden_size_128_learning_rate_0.001_batch_size_32_hidden_layers_2': {'val_f1': 0.8322621566978784},
  'hidden_size_128_learning_rate_0.001_batch_size_32_hidden_layers_3': {'val_f1': 0.8635798813945034}},
 {'hidden_size': 64,
  'learning_rate': 0.001,
  'batch_size': 16,
  'hidden_layers': 2},
 0.9035401377790354)

Here ends the Grid Search 
---

## 🚀✨ **Main Training Loop** ✨🚀


## 🧪 Testing

In [129]:
# Load model
best_model.eval()

# Build test sequences with static features
def build_test_sequences_with_static(df, window, stride, temporal_cols, static_cols):
    X_temporal = []
    X_static = []
    sample_indices = []
    
    for sid in df["sample_index"].unique():
        temp_temporal = df[df["sample_index"] == sid][temporal_cols].values.astype("float32")
        if np.isnan(temp_temporal).any():
            temp_df = pd.DataFrame(temp_temporal, columns=temporal_cols)
            temp_temporal = temp_df.ffill().bfill().fillna(0).values.astype("float32")
        
        temp_static = df[df["sample_index"] == sid][static_cols].values[0].astype("float32")
        if np.isnan(temp_static).any():
            temp_static = np.nan_to_num(temp_static, nan=0.0)
        
        remainder = len(temp_temporal) % window
        if remainder > 0:
            pad_len = window - remainder
            padding = np.zeros((pad_len, temp_temporal.shape[1]), dtype="float32")
            temp_temporal = np.concatenate((temp_temporal, padding), axis=0)
        
        idx = 0
        while idx + window <= len(temp_temporal):
            X_temporal.append(temp_temporal[idx:idx + window])
            X_static.append(temp_static)
            sample_indices.append(sid)
            idx += stride
    
    return np.array(X_temporal), np.array(X_static), sample_indices


# Build test sequences
X_test_temporal, X_test_static, test_sample_indices = build_test_sequences_with_static(
    df_public_test, WINDOW_SIZE, STRIDE, temporal_feature_cols, static_feature_cols
)

# Make predictions
final_predictions = []
with torch.no_grad():
    for sid in df_public_test["sample_index"].unique():
        mask = np.array(test_sample_indices) == sid
        if not mask.any():
            continue
        
        user_temporal = torch.tensor(X_test_temporal[mask], dtype=torch.float32).to(device)
        user_static = torch.tensor(X_test_static[mask], dtype=torch.float32).to(device)
        
        logits = best_model(user_temporal, user_static)
        probabilities = torch.nn.functional.softmax(logits, dim=1)
        mean_probs = torch.mean(probabilities, dim=0)
        pred_class = torch.argmax(mean_probs).item()
        final_predictions.append((sid, pred_class))

# Map to labels and save
inverse_label_mapping = {v: k for k, v in label_mapping.items()}
submission_df = pd.DataFrame({
    "sample_index": [f"{int(sid):03d}" for sid, _ in final_predictions],
    "label": [inverse_label_mapping[pred] for _, pred in final_predictions]
})
submission_df.to_csv("submission116.csv", index=False)
print("submission.csv created")

submission.csv created


In [128]:
best_epoch = np.argmax(training_history['val_f1']) + 1
print(f"Best epoch: {best_epoch} (Val F1: {training_history['val_f1'][best_epoch-1]:.4f})")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(training_history['train_loss'], label='Train')
ax1.plot(training_history['val_loss'], label='Val')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True)

ax2.plot(training_history['train_f1'], label='Train')
ax2.plot(training_history['val_f1'], label='Val')
ax2.axvline(best_epoch-1, color='r', linestyle='--', alpha=0.5, label=f'Best: {best_epoch}')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('F1 Score')
ax2.legend()
ax2.grid(True)
plt.tight_layout()
plt.show()


NameError: name 'training_history' is not defined

In [ ]:
def plotting(model_with_static, val_loader_static):
    # Confusion Matrix on Validation Set
    # Load best model if not already loaded
    model_with_static.load_state_dict(torch.load("models/lstm_static_best.pt"))
    model_with_static.eval()
    
    # Get predictions on validation set
    val_predictions = []
    val_targets = []
    
    with torch.no_grad():
        for inputs_temporal, inputs_static, targets in val_loader_static:
            inputs_temporal = inputs_temporal.to(device)
            inputs_static = inputs_static.to(device)
            targets = targets.to(device)
            
            logits = model_with_static(inputs_temporal, inputs_static)
            predictions = logits.argmax(dim=1)
            
            val_predictions.extend(predictions.cpu().numpy())
            val_targets.extend(targets.cpu().numpy())
    
    # Convert to numpy arrays
    val_predictions = np.array(val_predictions)
    val_targets = np.array(val_targets)
    
    # Create confusion matrix
    cm = confusion_matrix(val_targets, val_predictions)
    
    # Get class names from label mapping
    inverse_label_mapping = {v: k for k, v in label_mapping.items()}
    class_names = [inverse_label_mapping[i] for i in range(len(inverse_label_mapping))]
    
    # Plot confusion matrix
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names,
                cbar_kws={'label': 'Count'})
    plt.title('Confusion Matrix (Validation Set)', fontsize=16, pad=20)
    plt.ylabel('True Label', fontsize=14)
    plt.xlabel('Predicted Label', fontsize=14)
    plt.tight_layout()
    plt.show()
    
    # Print classification metrics
    print("\n" + "="*60)
    print("CLASSIFICATION METRICS (Validation Set)")
    print("="*60)
    print(f"Accuracy: {accuracy_score(val_targets, val_predictions):.4f}")
    print(f"Weighted F1: {f1_score(val_targets, val_predictions, average='macro'):.4f}")
    print(f"Precision: {precision_score(val_targets, val_predictions, average='macro', zero_division=0):.4f}")
    print(f"Recall: {recall_score(val_targets, val_predictions, average='macro', zero_division=0):.4f}")
    
    # Per-class metrics
    per_class_f1 = f1_score(val_targets, val_predictions, average=None, zero_division=0)
    per_class_precision = precision_score(val_targets, val_predictions, average=None, zero_division=0)
    per_class_recall = recall_score(val_targets, val_predictions, average=None, zero_division=0)
    
    print("\nPer-class metrics:")
    print(f"{'Class':<15} {'Precision':<12} {'Recall':<12} {'F1-Score':<12}")
    print("-" * 60)
    for i, class_name in enumerate(class_names):
        print(f"{class_name:<15} {per_class_precision[i]:<12.4f} {per_class_recall[i]:<12.4f} {per_class_f1[i]:<12.4f}")
    print("="*60)
plotting(best_model, val_loader_static)

In [ ]:
submission_df['label'].value_counts()

In [ ]:
# 4 epochs, early stopped, fixed L2 reg and LSTM ()
# no_pain      1000
# low_pain      198
# high_pain     126
# Name: count, dtype: int64

In [ ]:
# 35 epochs (netowrk width = 256) - f1 0.92
# no_pain      1035
# low_pain      213
# high_pain      76
# Name: count, dtype: int64

In [ ]:
# 17 epochs (batch size 32) - f1 0.82
# label
# no_pain      882
# high_pain    301
# low_pain     141
# Name: count, dtype: int64

In [ ]:
# 15 epochs (early stopped)
# label
# no_pain      953
# high_pain    197
# low_pain     174
# Name: count, dtype: int64

In [ ]:
# 10 epoch training result
# label
# no_pain      910
# high_pain    229
# low_pain     185
# Name: count, dtype: int64